# Feature-based deep-learning classification (CategoryEmbedding, GANDALF)

Fold-based cross-validation of two pytorch_tabular models on the 60 summary features, for every burst length (B1–B4) and correction type (uncorrected, rotdaily, rotbasal). Folds come from `indices_cv_fold_lookup.csv` (`04_build_cv_fold_lookup`): train = `fold != f`, test = `fold == f`, and all sub-windows of a parent burst share a fold.

Early stopping and checkpoint selection use a grouped, stratified split of the **training folds only** (`early_stopping_split`), so the test fold is never seen before the final evaluation.

Each run writes three outputs for the behaviour target (the activity target is also fitted, but its outputs are not saved; see the save cell):
- per-fold metrics (`*_features_dl_metrics.csv`)
- per-fold confusion matrices (`*_features_dl_confusion.csv`)
- per-window out-of-fold predictions with class probabilities (`*_features_dl_oof_predictions.csv`), carrying `Ind_ID`, `burst_id` and `fold`

Metrics are computed with the same `compute_metrics` helper as `05a`, `05c` and `05d`.

In [ ]:
import gc
import glob
import os
import time
import traceback

import numpy as np
import pandas as pd
import torch
from pytorch_tabular import TabularModel
from pytorch_tabular.config import (
    DataConfig,
    OptimizerConfig,
    TrainerConfig,
)
from pytorch_tabular.models import (
    CategoryEmbeddingModelConfig,
    GANDALFConfig,
)
from pytorch_tabular.models.common.heads import LinearHeadConfig
from pytorch_tabular.utils import get_balanced_sampler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder

## Shared helpers
Identical in `05a`, `05b`, `05c` and `05d`, so every model family uses the same metric definitions and output format.

In [ ]:
N_FEATURES = 60  # guards against ID columns leaking into X
TEST_ID_COLS = ["Ind_ID", "new_burst", "burst_id", "fold"]
LEAD_COLS = ["burst", "correction_type", "target", "fold", "model"]


def compute_metrics(y_true, y_pred, proba, n_classes, is_binary):
    """Identical metric definitions for every model family.

    Args:
        y_true (np.ndarray): Encoded true labels
        y_pred (np.ndarray): Encoded predicted labels
        proba (np.ndarray): Predicted class probabilities, one column per class
        n_classes (int): Number of classes in the target
        is_binary (bool): Whether this is binary classification

    Returns:
        dict: Global metrics (weighted and macro averages)
    """
    labels = np.arange(n_classes)
    try:
        roc = (
            roc_auc_score(y_true, proba[:, 1])
            if is_binary
            else roc_auc_score(
                y_true, proba, multi_class="ovr", average="weighted", labels=labels
            )
        )
    except ValueError:
        roc = np.nan
    return {
        "roc_auc": roc,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "f1_score": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "precision": precision_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_macro": f1_score(
            y_true, y_pred, average="macro", labels=labels, zero_division=0
        ),
        "precision_macro": precision_score(
            y_true, y_pred, average="macro", labels=labels, zero_division=0
        ),
        "recall_macro": recall_score(
            y_true, y_pred, average="macro", labels=labels, zero_division=0
        ),
    }


def confusion_rows(y_true, y_pred, class_names, model_name):
    """Fixed CxC confusion matrix in long format, so folds can be pooled."""
    n = len(class_names)
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(n))
    return [
        {
            "model": model_name,
            "actual_label": class_names[i],
            "predicted_label": class_names[j],
            "count": int(cm[i, j]),
        }
        for i in range(n)
        for j in range(n)
    ]


def prediction_rows(test_ids, y_true, y_pred, proba, class_names, model_name):
    """One row per test window: IDs, labels and class probabilities."""
    out = test_ids.reset_index(drop=True).copy()
    out.insert(0, "model", model_name)
    names = np.asarray(class_names, dtype=object)
    out["actual_label"] = names[np.asarray(y_true, dtype=int)]
    out["predicted_label"] = names[np.asarray(y_pred, dtype=int)]
    for i, cls in enumerate(class_names):
        out[f"prob_{cls}"] = proba[:, i]
    return out


def finalise(frames, lead_cols):
    """Concatenate per-fold frames and move the identifying columns to the front."""
    frames = [d for d in frames if d is not None and not d.empty]
    if not frames:
        return pd.DataFrame(columns=lead_cols)
    df = pd.concat(frames, ignore_index=True)
    return df[lead_cols + [c for c in df.columns if c not in lead_cols]]


## Early-stopping split and batch-size guard

In [ ]:
SEED = 42
ES_SPLITS = 7  # holds out ~1/7 (~14%) of the training parents for early stopping


def early_stopping_split(y_train, groups_train, n_classes, seed=SEED, max_tries=20):
    """Grouped, stratified split of the TRAINING folds only. The test fold is
    never seen during training, early stopping or checkpoint selection.

    Re-draws (seed, seed+1, ...) until every class is in the fitting split,
    because pytorch_tabular silently drops the output of an unseen class. A
    class may still be absent from the early-stopping subset; that only removes
    it from the stopping loss.

    Args:
        y_train (np.ndarray): Encoded training labels
        groups_train (np.ndarray): Parent-burst ID per training window
        n_classes (int): Number of classes in the target
        seed (int): First random_state to try
        max_tries (int): Number of seeds to try before giving up

    Returns:
        tuple: (fit_idx, es_idx) positional indices into the training set
    """
    y_train, groups_train = np.asarray(y_train), np.asarray(groups_train)
    if len(np.unique(y_train)) < n_classes:
        raise ValueError("A class is missing from the training folds themselves")
    for k in range(max_tries):
        sgkf = StratifiedGroupKFold(
            n_splits=ES_SPLITS, shuffle=True, random_state=seed + k
        )
        fit_idx, es_idx = next(sgkf.split(np.zeros(len(y_train)), y_train, groups_train))
        if len(np.unique(y_train[fit_idx])) == n_classes:
            if k > 0:
                print(f"  early-stopping split re-drawn {k} time(s) to keep all classes")
            return fit_idx, es_idx
    raise ValueError("No early-stopping split keeps every class in the fitting split")


def safe_batch_size(*sizes, bs=64):
    """Smallest batch size >= bs that leaves no final batch of a single sample in
    any of the given sets. A one-sample batch raises an IndexError inside
    pytorch_tabular (this previously dropped some folds)."""
    while any(n % bs == 1 for n in sizes):
        bs += 1
    return bs


## Model functions

In [ ]:
def features_dl_models(
    X_train,
    X_test,
    y_train,
    y_test,
    class_names,
    groups_train,
    test_ids,
    use_class_weight=True,
    is_binary=False,
    seed=SEED,
):
    """Train and evaluate CategoryEmbedding and GANDALF on one CV fold.

    Args:
        X_train (pd.DataFrame): Training features
        X_test (pd.DataFrame): Test features
        y_train (np.ndarray): Encoded training labels
        y_test (np.ndarray): Encoded test labels
        class_names (list): Original class names, ordered by encoded label
        groups_train (np.ndarray): Parent-burst ID per training window, for the
            grouped early-stopping split
        test_ids (pd.DataFrame): TEST_ID_COLS for the test windows, in X_test order
        use_class_weight (bool): Whether to balance training batches via
            get_balanced_sampler (inverse class frequency, mirroring
            class_weight="balanced" in the ML pipeline). Only the training
            dataloader is resampled, so the early-stopping loss stays on the
            natural class distribution.
        is_binary (bool): Whether this is binary classification
        seed (int): Seed for the early-stopping split and pytorch_tabular

    Returns:
        tuple: (metrics_df, confusion_df, predictions_df). A model that fails
        to train is skipped (no placeholder row).
    """
    num_classes = len(class_names)

    train_df = X_train.copy().reset_index(drop=True)
    train_df["target"] = np.asarray(y_train)
    test_df = X_test.copy().reset_index(drop=True)
    test_df["target"] = np.asarray(y_test)

    fit_idx, es_idx = early_stopping_split(y_train, groups_train, num_classes, seed)
    fit_df = train_df.iloc[fit_idx].reset_index(drop=True)
    es_df = train_df.iloc[es_idx].reset_index(drop=True)
    bs = safe_batch_size(len(fit_df), len(es_df), len(test_df))

    data_config = DataConfig(
        target=["target"],
        continuous_cols=list(X_train.columns),
        categorical_cols=[],
        pin_memory=True,
        num_workers=0,
    )
    trainer_config = TrainerConfig(
        auto_lr_find=True,
        batch_size=bs,
        max_epochs=100,
        early_stopping="valid_loss",
        early_stopping_mode="min",
        early_stopping_min_delta=0.001,
        early_stopping_patience=10,
        checkpoints="valid_loss",
        load_best=True,
        track_grad_norm=2,
        progress_bar="none",
        accelerator="gpu",
        seed=seed,  # 42 is already the default; explicit only
    )
    head_config = LinearHeadConfig(
        layers="",
        dropout=0.1,
        initialization="kaiming",
    ).__dict__
    optimizer_config = OptimizerConfig(
        lr_scheduler="CosineAnnealingWarmRestarts",
        lr_scheduler_params={"T_0": 100, "T_mult": 1, "eta_min": 1e-5},
    )
    common_params = {
        "task": "classification",
        "head": "LinearHead",
        "head_config": head_config,
    }
    model_list = [
        CategoryEmbeddingModelConfig(layers="1024-512-256", **common_params),
        GANDALFConfig(gflu_stages=15, learnable_sparsity=False, **common_params),
    ]
    train_sampler = (
        get_balanced_sampler(fit_df["target"].to_numpy()) if use_class_weight else None
    )

    metrics, conf, preds = [], [], []
    for model_config in model_list:
        name = model_config.__class__.__name__.replace("Config", "")
        try:
            print(f"  Training {name}")
            t0 = time.time()
            model = TabularModel(
                data_config=data_config,
                model_config=model_config,
                optimizer_config=optimizer_config,
                trainer_config=trainer_config,
                verbose=False,
            )
            model.fit(train=fit_df, validation=es_df, train_sampler=train_sampler)
            pred_df = model.predict(test_df)
            y_pred = pred_df["prediction"].to_numpy().astype(int)
            # Probability columns are named "<class index>_probability"
            prob_cols = sorted(
                [c for c in pred_df.columns if c.endswith("_probability")],
                key=lambda c: int(c.split("_")[0]),
            )
            proba = pred_df[prob_cols].to_numpy()
            assert proba.shape[1] == num_classes
            metrics.append(
                {
                    "model": name,
                    **compute_metrics(y_test, y_pred, proba, num_classes, is_binary),
                    "time": time.time() - t0,
                }
            )
            conf += confusion_rows(y_test, y_pred, class_names, name)
            preds.append(
                prediction_rows(test_ids, y_test, y_pred, proba, class_names, name)
            )
            del model
        except Exception as e:
            print(f"!! Error training/evaluating {name}: {e!s}")
            print(traceback.format_exc())
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    preds = pd.concat(preds, ignore_index=True) if preds else pd.DataFrame()
    return pd.DataFrame(metrics), pd.DataFrame(conf), preds

In [ ]:
ID_COLS = ["Ind_ID", "new_burst", "burst_id", "behaviour", "activity", "fold"]


def process_parquet_file(file_path, fold_lookup, use_class_weight=True, max_folds=None):
    """Run fold-based CV on one parquet file for both the binary (activity)
    and multiclass (behaviour) targets.

    Args:
        file_path (str): Path to the parquet file
        fold_lookup (pd.DataFrame): Fold lookup table with columns
            burst, Ind_ID, new_burst, burst_id, behaviour, activity, fold
        use_class_weight (bool): Whether to balance training batches via
            get_balanced_sampler
        max_folds (int, optional): Run only the first max_folds folds (smoke test)

    Returns:
        dict: binary_/multiclass_ metrics, conf and pred DataFrames
    """
    burst = os.path.basename(os.path.dirname(file_path))
    correction_type = os.path.basename(file_path).split("_")[-1].split(".")[0]
    print(f"Processing {correction_type} for {burst}")

    features = pd.read_parquet(file_path)  # features + Ind_ID + new_burst
    m = fold_lookup.loc[fold_lookup["burst"] == burst.lower(), ID_COLS]
    df = features.merge(m, on=["Ind_ID", "new_burst"], how="inner").reset_index(
        drop=True
    )  # inner drops Remove/unlabelled
    X = df.drop(columns=ID_COLS)
    assert X.shape[1] == N_FEATURES, f"Unexpected feature columns: {list(X.columns)}"
    folds = sorted(df["fold"].unique())[:max_folds]
    # Parent burst per window, for the grouped early-stopping split
    groups = (df["Ind_ID"].astype(str) + "_" + df["burst_id"].astype(str)).to_numpy()

    out = {t: {"metrics": [], "conf": [], "pred": []} for t in ("binary", "multiclass")}
    for is_binary, col, tgt, key in [
        (True, "activity", "Activity", "binary"),
        (False, "behaviour", "Behaviour", "multiclass"),
    ]:
        enc = LabelEncoder()
        y_enc = enc.fit_transform(df[col])
        # Encode "Active" as 1 so binary ROC AUC uses the active class
        if (
            is_binary
            and set(enc.classes_) == {"Active", "Inactive"}
            and enc.transform(["Active"])[0] != 1
        ):
            enc.classes_ = enc.classes_[::-1]
            y_enc = enc.transform(df[col])
        class_names = list(enc.classes_)

        for f in folds:
            te = (df["fold"] == f).to_numpy()
            print(f"    Running {tgt} classification, fold {f}")
            met, conf, pred = features_dl_models(
                X[~te],
                X[te],
                y_enc[~te],
                y_enc[te],
                class_names,
                groups_train=groups[~te],
                test_ids=df.loc[te, TEST_ID_COLS],
                use_class_weight=use_class_weight,
                is_binary=is_binary,
            )
            for d in (met, conf, pred):
                if not d.empty:
                    d["correction_type"] = correction_type
                    d["burst"] = burst
                    d["target"] = tgt
                    d["fold"] = f
            out[key]["metrics"].append(met)
            out[key]["conf"].append(conf)
            out[key]["pred"].append(pred)

            # Free GPU memory between folds
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    return {
        f"{key}_{kind}": finalise(out[key][kind], LEAD_COLS)
        for key in ("binary", "multiclass")
        for kind in ("metrics", "conf", "pred")
    }

In [ ]:
def wrap_run_feat_dl(
    folder_location,
    fold_id_path="../data/temp/indices_cv_fold_lookup.csv",
    correction_filters=None,
    use_class_weight=True,
):
    """Process all parquet files in a folder.

    Args:
        folder_location (str): Path to folder containing parquet files
        fold_id_path (str): Path to CSV file containing fold IDs
        correction_filters (list, optional): List of correction types to include
        use_class_weight (bool): Whether to balance training batches via
            get_balanced_sampler

    Returns:
        dict: Dictionary with combined results DataFrames
    """
    filetype = folder_location.split("/")[-2]
    print(f"DL training started for {filetype}")

    # Find all parquet files
    all_files = [
        os.path.normpath(f).replace("\\", "/")
        for f in glob.glob(folder_location + "**/*.parquet", recursive=True)
    ]

    # Filter by correction type
    if correction_filters is not None:
        all_files = [
            f
            for f in all_files
            if any(corr.lower() in f.lower() for corr in correction_filters)
        ]

    print(f"Found {len(all_files)} parquet files")

    fold_lookup = pd.read_csv(fold_id_path)

    # Process all files and combine results
    all_results = [
        process_parquet_file(f, fold_lookup, use_class_weight=use_class_weight)
        for f in all_files
    ]
    keys = [
        f"{t}_{k}"
        for t in ("binary", "multiclass")
        for k in ("metrics", "conf", "pred")
    ]
    return {k: pd.concat([r[k] for r in all_results], ignore_index=True) for k in keys}

## Smoke test (one file, one fold)
Set `RUN_SMOKE_TEST = True` to run it before the overnight run; it is skipped otherwise. Checks that the OOF output has one row per test window, the probabilities sum to 1 and `burst_id` is filled.

In [ ]:
# Set to True to run the smoke test interactively; False skips it in Run All / nbconvert
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    fold_lookup = pd.read_csv("../data/temp/indices_cv_fold_lookup.csv")
    smoke = process_parquet_file(
        "../data/raw/features/Burst_2/annotated_features_burst_2_uncorrected.parquet",
        fold_lookup,
        max_folds=1,
    )
    for key in ("binary", "multiclass"):
        pred = smoke[f"{key}_pred"]
        prob = pred.filter(like="prob_")
        n_test = (
            (fold_lookup["burst"] == "burst_2") & (fold_lookup["fold"] == pred["fold"].iloc[0])
        ).sum()
        print(
            f"{key}: models={sorted(pred['model'].unique())}, "
            f"rows per model={pred.groupby('model').size().tolist()} (test windows: {n_test}), "
            f"max |prob sum - 1|={(prob.sum(axis=1) - 1).abs().max():.2e}, "
            f"burst_id missing={pred['burst_id'].isna().sum()}"
        )
    display(smoke["multiclass_metrics"])


## Run over all bursts and correction types

In [ ]:
fold_id_path = "../data/temp/indices_cv_fold_lookup.csv"
correction_types_to_process = ["uncorrected", "rotdaily", "rotbasal"]
folder_locations = [
    "../data/raw/features/Burst_1/",
    "../data/raw/features/Burst_2/",
    "../data/raw/features/Burst_3/",
    "../data/raw/features/Burst_4/",
]

In [ ]:
# Run the DL training for all files in the specified folders
all_burst_results = [
    wrap_run_feat_dl(
        folder, fold_id_path, correction_filters=correction_types_to_process
    )
    for folder in folder_locations
]

# Combine bursts into single dataframes
res = {
    k: pd.concat([r[k] for r in all_burst_results], ignore_index=True)
    for k in all_burst_results[0]
}

## Check for failed fits
A model that fails on a fold leaves no row, so every (dataset, target, model) should have 5 folds.

In [ ]:
for k in ("binary_metrics", "multiclass_metrics"):
    n = res[k].groupby(["burst", "correction_type", "model"])["fold"].nunique()
    print(f"{k}: {len(n)} (dataset, model) combinations for this target, {(n < 5).sum()} with missing folds")
    if (n < 5).any():
        print(n[n < 5])


## Save

In [ ]:
# Output to CSV files
out_files = {
    # Activity (active/inactive) outputs are not used in the paper and are not
    # shipped; uncomment to save them (create data/output/comparison_activity/ first)
    # "binary_metrics": "comparison_activity/activity_features_dl_metrics.csv",
    # "binary_conf": "comparison_activity/activity_features_dl_confusion.csv",
    # "binary_pred": "comparison_activity/activity_features_dl_oof_predictions.csv",
    "multiclass_metrics": "comparison_behaviour/behaviour_features_dl_metrics.csv",
    "multiclass_conf": "comparison_behaviour/behaviour_features_dl_confusion.csv",
    "multiclass_pred": "comparison_behaviour/behaviour_features_dl_oof_predictions.csv",
}
for k, path in out_files.items():
    res[k].to_csv(f"../data/output/{path}", index=False)
    print(f"Saved {len(res[k])} rows -> {path}")